In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [4]:
df_ground_truth.head(10)

,question,document
0,I just found this course a bit late — can I st...,74eb249bbf
1,Is it okay to join the course after it has alr...,74eb249bbf
2,"If I enroll now, will I still be able to take ...",74eb249bbf
3,I’m new here and missed the beginning — can I ...,74eb249bbf
4,"Can I join the course late, and what do I need...",74eb249bbf
5,Do I actually need a confirmation email after ...,977bf7786c
6,"If I registered for LLM Zoomcamp, when should ...",977bf7786c
7,Can I begin the lessons and submit homework ev...,977bf7786c
8,Is the signup tied to any official participant...,977bf7786c
9,What is the point of registering for the cours...,977bf7786c


In [5]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [6]:
ground_truth[10]

{'question': "Where do students watch the Office Hours live if they don't have the Zoom link?",
 'document': '489dd1c9d9'}

In [8]:
import sys
sys.path.append('../01-agentic-rag')


In [9]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [10]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson

In [14]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [15]:
q = ground_truth[0]
q

{'question': 'I just found this course a bit late — can I still sign up and join in?',
 'document': '74eb249bbf'}

In [16]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [17]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only ge

In [18]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
69d122f12e == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [19]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [20]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [21]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course a bit late — can I still sign up and join in?


[1, 0, 0, 0, 0]

In [22]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

How do we ask questions during the live/workshop sessions instead of posting in chat?


[0, 1, 0, 0, 0]

In [23]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [24]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/805 [00:00<?, ?it/s]

In [25]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [26]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [27]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [28]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/805 [00:00<?, ?it/s]

In [32]:
sample = relevance_total[:20]

In [33]:
sample

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [34]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.75

In [35]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [36]:
hit_rate(relevance)

0.831055900621118

In [37]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6083333333333334

In [38]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [39]:
mrr(sample)

0.6083333333333334

In [40]:
mrr(relevance)

0.712070393374741

In [41]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [42]:
evaluate(ground_truth, text_search)

  0%|          | 0/805 [00:00<?, ?it/s]

{'hit_rate': 0.831055900621118, 'mrr': 0.712070393374741}

In [43]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [44]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/805 [00:00<?, ?it/s]

{'hit_rate': 0.8608695652173913, 'mrr': 0.7472877846790886}

In [45]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [46]:
def text_search_v3(query):
    boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.1}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

evaluate(ground_truth, text_search_v3)

  0%|          | 0/805 [00:00<?, ?it/s]

{'hit_rate': 0.9540372670807453, 'mrr': 0.8548447204968946}

In [47]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/805 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.884472049689441, 'mrr': 0.789130434782609}


  0%|          | 0/805 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8881987577639752, 'mrr': 0.7911387163561078}


  0%|          | 0/805 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.831055900621118, 'mrr': 0.712070393374741}


  0%|          | 0/805 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.7950310559006211, 'mrr': 0.6799999999999997}


  0%|          | 0/805 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7677018633540372, 'mrr': 0.6497308488612832}


In [48]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [49]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/805 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/805 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


  0%|          | 0/805 [00:00<?, ?it/s]

... (62 outputs left out)

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/805 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/805 [00:00<?, ?it/s]

In [50]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.963975,0.855031
3,1.0,2.0,0.1,0.954037,0.854845
19,2.0,4.0,0.2,0.954037,0.854845
35,5.0,10.0,0.5,0.954037,0.854845
4,1.0,2.0,0.2,0.950311,0.853209
18,2.0,4.0,0.1,0.951553,0.853126
34,5.0,10.0,0.2,0.951553,0.853064
33,5.0,10.0,0.1,0.952795,0.852692
8,1.0,4.0,0.5,0.959006,0.852609
6,1.0,4.0,0.1,0.961491,0.851781
